# SLR-Engine V04: Reviewing AI-Written Code With SURF AI-HUB

## Workflow overview

1. Confirm that the notebook is running on the intended `prisma-env` kernel.
2. Validate the local SLR-Engine checkout and run its test suite before changing project data.
3. Create or reuse the review project for **The Problem With Using AI to Review AI-Written Code**.
4. Persist a review protocol with research questions, eligibility criteria, and an explicit human-decision boundary.
5. Store literal search queries, retrieve source records, deduplicate them, and prepare screening batches.
6. Ask a SURF AI-HUB Qwen 2.5 Instruct model with at least 32B parameters for a bounded advisory recommendation.
7. Keep every inclusion or exclusion decision under human control and retain the project audit trail.

The SURF AI-HUB step sends only the selected candidate title and abstract to the configured service. It does not submit repository code, does not write a screening decision, and may consume service quota.

In [1]:
from pathlib import Path
import platform
import sys

expected_kernel = 'prisma-env'
python_path = Path(sys.executable)
kernel_description = {
    'Python executable': str(python_path),
    'Python version': platform.python_version(),
    'Environment prefix': sys.prefix,
}

for label, value in kernel_description.items():
    print(f'{label}: {value}')

if expected_kernel not in str(python_path).lower() and expected_kernel not in sys.prefix.lower():
    raise RuntimeError(
        'Wrong kernel loaded. Select the prisma-env kernel, then restart and run this notebook again.'
    )

print(f'Kernel check passed: {expected_kernel}')

Python executable: C:\Users\PROMET02\anaconda3\envs\prisma-env\python.exe
Python version: 3.11.15
Environment prefix: C:\Users\PROMET02\anaconda3\envs\prisma-env
Kernel check passed: prisma-env


## 1. Verify the local repository

The test suite validates the repository before this notebook creates a review project. It does not contact external literature sources.

In [2]:
import subprocess

repo_root = Path.cwd().resolve()
required_paths = [
    repo_root / 'README.md',
    repo_root / 'scripts' / '00_init_project.py',
    repo_root / 'slr_engine',
    repo_root / 'tests',
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Open this notebook from the SLR-Engine repository root. Missing: ' + ', '.join(missing_paths)
    )

test_result = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q'],
    cwd=repo_root,
    text=True,
    capture_output=True,
)
print(test_result.stdout)
if test_result.returncode:
    print(test_result.stderr)
    raise RuntimeError(f'Repository tests failed with exit code {test_result.returncode}.')

print('Repository test suite passed.')

...........                                                              [100%]
11 passed in 0.98s

Repository test suite passed.


## 2. Define the review protocol

**Problem:** AI-generated code can look plausible while hiding security, correctness, licensing, or maintainability defects. An AI reviewer may repeat the same failure mode, accept persuasive but unsupported explanations, or lack the context needed to validate a change.

**Research questions:**
- What empirical evidence measures the effectiveness or failure modes of AI systems reviewing AI-generated code?
- Which review tasks, evaluation datasets, and human oversight mechanisms are reported?
- Which risks are documented, including hallucinated findings, missed defects, automation bias, and limited reproducibility?

The protocol below configures SLR-Engine for a local project only. It does not search external sources.

In [3]:
from slr_engine.store import ProjectConfig, init_project

project_id = 'Willma_SLR'
project_root = repo_root / 'projects' / project_id
topic = 'The Problem With Using AI to Review AI-Written Code'

if not project_root.exists():
    init_project(repo_root / 'projects', project_id, topic=topic)
    print(f'Created project: {project_root}')
else:
    print(f'Reusing existing project: {project_root}')

config = ProjectConfig.load(project_root)
config.topic = topic
config.aim = (
    'Synthesize empirical evidence about the reliability, risks, and human oversight '
    'needed when AI reviews AI-written code.'
)
config.research_questions = [
    'What evidence evaluates AI review of AI-written code?',
    'Which defects, risks, and failure modes are missed or introduced?',
    'Which human oversight and reproducibility practices are reported?',
]
config.inclusion = [
    {'id': 'I1', 'text': 'Empirical study, benchmark, or systematic evaluation.'},
    {'id': 'I2', 'text': 'Evaluates AI-assisted or AI-driven code review, code analysis, or defect detection.'},
    {'id': 'I3', 'text': 'Reports methods, data, outcomes, or documented limitations.'},
]
config.exclusion = [
    {'id': 'E1', 'text': 'Opinion-only content without a described evaluation method.'},
    {'id': 'E2', 'text': 'General code generation without a review or evaluation component.'},
    {'id': 'E3', 'text': 'Duplicate publication or insufficient bibliographic metadata.'},
]
config.seed_examples = {
    'include': ['Replace with verified seed papers after human review.'],
    'exclude': ['Marketing claims about AI code review without an evaluation.'],
}
config.llm = {'provider': 'agent'}
config.save(project_root)

print(f'Protocol saved: {project_root / "project.yaml"}')

Created project: D:\OneDrive - Hogeschool Rotterdam\1_CURRENT_CODE\FLOWISE_DEEP_RESEARCH_GOOGLE\GOOGLE_API_KEY\GOOGLE_SCHOLAR\SLR-Engine\projects\Willma_SLR
Protocol saved: D:\OneDrive - Hogeschool Rotterdam\1_CURRENT_CODE\FLOWISE_DEEP_RESEARCH_GOOGLE\GOOGLE_API_KEY\GOOGLE_SCHOLAR\SLR-Engine\projects\Willma_SLR\project.yaml


In [4]:
import pandas as pd

protocol_rows = [
    ('Topic', config.topic),
    ('Aim', config.aim),
    ('Research questions', '\n'.join(config.research_questions)),
    ('Inclusion criteria', '\n'.join(item['id'] + ': ' + item['text'] for item in config.inclusion)),
    ('Exclusion criteria', '\n'.join(item['id'] + ': ' + item['text'] for item in config.exclusion)),
    ('Decision provider', config.judgment_provider()),
]
protocol_table = pd.DataFrame(protocol_rows, columns=['Protocol item', 'Value'])
pd.set_option('display.max_colwidth', 1_000)
display(protocol_table)

,Protocol item,Value
0,Topic,The Problem With Using AI to Review AI-Written Code
1,Aim,"Synthesize empirical evidence about the reliability, risks, and human oversight needed when AI reviews AI-written code."
2,Research questions,"What evidence evaluates AI review of AI-written code?\nWhich defects, risks, and failure modes are missed or introduced?\nWhich human oversight and reproducibility practices are reported?"
3,Inclusion criteria,"I1: Empirical study, benchmark, or systematic evaluation.\nI2: Evaluates AI-assisted or AI-driven code review, code analysis, or defect detection.\nI3: Reports methods, data, outcomes, or documented limitations."
4,Exclusion criteria,E1: Opinion-only content without a described evaluation method.\nE2: General code generation without a review or evaluation component.\nE3: Duplicate publication or insufficient bibliographic metadata.
5,Decision provider,agent


## 3. Run a bounded evidence search

The queries below are deliberately narrow and are saved in the project audit trail. The run is capped at 25 records per source so the returned evidence can be inspected before deduplication and screening.

In [5]:
import json
import sqlite3

query_texts = {
    'concepts.yaml': """concepts:
  - id: C1
    preferred: AI-assisted code review
    synonyms:
      - AI code review
      - LLM code review
      - automated code review
  - id: C2
    preferred: AI-generated code
    synonyms:
      - generated code
      - code generation
      - large language model code
""",
    'openalex.txt': (
        '("AI code review" OR "LLM code review" OR "automated code review") '
        'AND ("AI-generated code" OR "generated code" OR "code generation")'
    ),
    'arxiv.txt': (
        '(all:"code review" OR all:"automated code review") '
        'AND (all:"large language model" OR all:"AI generated code")'
    ),
    'semantic_scholar.txt': (
        '("AI code review" OR "LLM code review") '
        'AND ("AI-generated code" OR "code generation")'
    ),
    'crossref.json': json.dumps(
        {
            'filter': {'type': 'journal-article', 'has-abstract': 'true'},
            'query.bibliographic': [
                'AI code review',
                'large language model generated code',
            ],
        },
        indent=2,
    ),
}

for filename, content in query_texts.items():
    (project_root / 'queries' / filename).write_text(content + '\n', encoding='utf-8')

search_result = subprocess.run(
    [
        sys.executable,
        'scripts/02_search_open.py',
        '--project',
        project_id,
        '--sources',
        'openalex,crossref,arxiv,semantic_scholar',
        '--max-records',
        '25',
        '--acknowledge-warnings',
    ],
    cwd=repo_root,
    text=True,
    capture_output=True,
 )
print(search_result.stdout)
if search_result.returncode:
    print(search_result.stderr)
    raise RuntimeError(f'External literature search failed with exit code {search_result.returncode}.')

with sqlite3.connect(project_root / 'project.db') as connection:
    evidence_table = pd.read_sql_query(
        """
        SELECT
            records.title,
            records.year,
            records.first_author,
            records.venue,
            records.abstract,
            GROUP_CONCAT(DISTINCT source_hits.source) AS sources
        FROM records
        LEFT JOIN source_hits ON source_hits.record_id = records.id
        GROUP BY records.id
        ORDER BY records.id
        LIMIT 10
        """,
        connection,
    )

if evidence_table.empty:
    raise RuntimeError('The search completed but returned no records to review.')

display(evidence_table)

[warn] OpenAlex: no OPENALEX_API_KEY set. Long Boolean queries
       may fail with HTTP 400. Set OPENALEX_API_KEY in .env or
       openalex_api_key in project.yaml.
Validating queries...

[openalex]
  ok

[crossref]
  ok

[arxiv]
  ok

[semantic_scholar]
  ok

[run] openalex: query_id=openalex_20260815_211841
[done] openalex: 25 records
[run] crossref: query_id=crossref_20260815_211844
[done] crossref: 25 records
[run] arxiv: query_id=arxiv_20260815_211846
[done] arxiv: 25 records
[run] semantic_scholar: query_id=semantic_scholar_20260815_211846
  [silent-zero] semantic_scholar: 0 records ingested AFTER 1 request error(s) — this is likely a network or API failure, not a real null result
[done] semantic_scholar: 0 records

Search summary:
  openalex: 25
  crossref: 25
  arxiv: 25
  semantic_scholar: 0
  total: 75

Post-search sanity check...

BLOCKING:
  - semantic_scholar: returned 0 records but had HTTP errors during the run — likely network/API failure, not a real null result. Inve

,title,year,first_author,venue,abstract,sources
0,Towards automating code review at scale,2021,Hellendoorn,NaN,"As neural methods are increasingly used to support and automate software development tasks, code review is a natural next target. Yet, training models to imitate developers based on past code reviews is far from straightforward: reviews found in open-source projects vary greatly in quality, phrasing, and depth depending on the reviewer. In addition, changesets are often large, stretching the capacity of current neural models. Recent work reported modest success at predicting review resolutions, but largely side-stepped the above issues by focusing on small inputs where comments were already known to occur. This work examines the vision and challenges of automating code review at realistic scale. We collect hundreds of thousands of changesets across hundreds of projects that routinely conduct code review, many of which change thousands of tokens. We focus on predicting just the locations of comments, which are quite rare. By analyzing model performance and dataset statistics, we sho...",openalex
1,Fine-Tuning Large Language Models to Improve Accuracy and Comprehensibility of Automated Code Review,2024,Yu,ACM Transactions on Software Engineering and Methodology,"As code review is a tedious and costly software quality practice, researchers have proposed several machine learning-based methods to automate the process. The primary focus has been on accuracy, that is, how accurately the algorithms are able to detect issues in the code under review. However, human intervention still remains inevitable since results produced by automated code review are not 100% correct. To assist human reviewers in making their final decisions on automatically generated review comments, the comprehensibility of the comments underpinned by accurate localization and relevant explanations for the detected issues with repair suggestions is paramount. However, this has largely been neglected in the existing research. Large language models (LLMs) have the potential to generate code review comments that are more readable and comprehensible by humans, thanks to their remarkable processing and reasoning capabilities. However, even mainstream LLMs perform poorly in detect...",openalex
2,Diggit: Automated code review via software repository mining,2018,Chatley,NaN,"We present Diggit, a tool to automatically generate code review comments, offering design guidance on prospective changes, based on insights gained from mining historical changes in source code repositories. We describe how the tool was built and tuned for use in practice as we integrated Diggit into the working processes of an industrial development team. We focus on the developer experience, the constraints that had to be met in adapting academic research to produce a tool that was useful to developers, and the effectiveness of the results in practice.",openalex
3,From ChatGPT to ThreatGPT: Impact of Generative AI in Cybersecurity and Privacy,2023,Gupta,IEEE Access,"Undoubtedly, the evolution of Generative AI (GenAI) models has been the highlight of digital transformation in the year 2022. As the different GenAI models like ChatGPT and Google Bard continue to foster their complexity and capability, it’s critical to understand its consequences from a cybersecurity perspective. Several instances recently have demonstrated the use of GenAI tools in both the defensive and offensive side of cybersecurity, and focusing on the social, ethical and privacy implications this technology possesses. This research paper highlights the limitations, challenges, potential risks, and opportunities of GenAI in the domain of cybersecurity and privacy. The work presents the vulnerabilities of ChatGPT, which can be exploited by malicious users to exfiltrate malicious information bypassing the ethical constraints on the model. This paper demonstrates successful example attacks like Jailbreaks, reverse psychology

In [6]:
# Human-readable overview of the retrieved evidence before screening.
with sqlite3.connect(project_root / 'project.db') as connection:
    papers = pd.read_sql_query(
        """
        SELECT
            records.canonical_id,
            records.title,
            records.year,
            records.first_author,
            records.venue,
            records.abstract,
            records.url,
            GROUP_CONCAT(DISTINCT source_hits.source) AS sources
        FROM records
        LEFT JOIN source_hits ON source_hits.record_id = records.id
        GROUP BY records.id
        """,
        connection,
    )

relevance_terms = (
    'code review',
    'automated code review',
    'ai code review',
    'llm',
    'large language model',
    'generated code',
    'code generation',
    'llm-as-a-judge',
 )

def relevance_score(paper):
    text = f"{paper['title'] or ''} {paper['abstract'] or ''}".lower()
    return sum(term in text for term in relevance_terms)

papers['relevance_signals'] = papers.apply(relevance_score, axis=1)
papers['abstract_preview'] = papers['abstract'].fillna('').str.replace(r'\s+', ' ', regex=True).str.slice(0, 350)
papers.loc[papers['abstract_preview'].str.len() == 350, 'abstract_preview'] += '...'

paper_overview = (
    papers.sort_values(['relevance_signals', 'year', 'title'], ascending=[False, False, True])
    .loc[:, [
        'canonical_id',
        'title',
        'year',
        'first_author',
        'venue',
        'sources',
        'relevance_signals',
        'abstract_preview',
        'url',
    ]]
    .reset_index(drop=True)
 )

source_overview = (
    papers.assign(source=papers['sources'].fillna('unknown').str.split(','))
    .explode('source')
    .groupby('source', dropna=False)['canonical_id']
    .nunique()
    .rename('papers')
    .reset_index()
    .sort_values('papers', ascending=False)
 )

year_overview = (
    papers.groupby('year', dropna=False)['canonical_id']
    .nunique()
    .rename('papers')
    .reset_index()
    .sort_values('year', ascending=False)
 )

print(f'Retrieved papers after deduplication: {len(paper_overview)}')
display(source_overview)
display(year_overview)
display(paper_overview)

Retrieved papers after deduplication: 75


,source,papers
0,arxiv,25
1,crossref,25
2,openalex,25


,year,papers
8,2026,37
7,2025,14
6,2024,10
5,2023,7
4,2022,2
3,2021,2
2,2018,1
1,2013,1
0,2012,1


,canonical_id,title,year,first_author,venue,sources,relevance_signals,abstract_preview,url
0,rec_000074,"Evaluation of LLM-Based Software Engineering Tools: Practices, Challenges, and Future Directions",2026,Torun,cs.SE,arxiv,5,"Large Language Models (LLMs) are increasingly embedded in software engineering (SE) tools, powering applications such as code generation, automated code review, and bug triage. As these LLM-based AI for Software Engineering (AI4SE) systems transition from experimental prototypes to widely deployed tools, the question of what it means to evaluate th...",https://arxiv.org/pdf/2604.24621v1
1,rec_000010,Can LLMs Replace Human Evaluators? An Empirical Study of LLM-as-a-Judge in Software Engineering,2025,Wang,Proceedings of the ACM on software engineering.,openalex,5,"Recently, large language models (LLMs) have been deployed to tackle various software engineering (SE) tasks like code generation, significantly advancing the automation of SE tasks. However, assessing the quality of these LLM-generated code and text remains challenging. The commonly used Pass@k metric necessitates extensive unit tests and configure...",https://doi.org/10.1145/3728963
2,rec_000006,Automating Code Reviews with Simulink Code Inspector.,2012,Conrad,MBEES,openalex,5,"Safety standards such as DO-178B require source code reviews. Given the maturity of today‟s code generators, the effectiveness of manual reviews of automatically generated code is rather limited. This results in a strong desire to automate reviews of automatically generated code. This paper introduces Simulink Code Inspector TM , a novel tool to au...",https://www.mathworks.com/tagteam/71296_CEE+15.pdf
3,rec_000044,AI ASSISTANTS IN SOFTWARE DEVELOPMENT: ANALYSIS OF SECURITY RISKS IN GENERATED CODE,2026,Boichuk,Grail of Science,crossref,4,"This article examines the widespread adoption of AI coding assistants in software development and the security risks that accompany this shift. It analyzes characteristic vulnerability patterns in code generated by tools such as GitHub Copilot, Amazon CodeWhisperer, and similar large language model (LLM) based systems. The paper reviews quantitativ...",https://doi.org/10.36074/grail-of-science.01.05.2026.077
4,rec_000072,Evaluating LLM-Generated Code: A Benchmark and Developer Study,2026,Szych,cs.SE,arxiv,4,"Code generation is one of the tasks for which the use of Large Language Models is widely adopted and highly successful. Given this popularity, there are many benchmarks dedicated to code generation that can help select the best model. However, they primarily focus on measuring solution correctness, leaving other aspects, such as code quality and us...",https://arxiv.org/pdf/2605.09059v2
...,...,...,...,...,...,...,...,...,...
70,rec_000004,From ChatGPT to ThreatGPT: Impact of Generative AI in Cybersecurity and Privacy,2023,Gupta,IEEE Access,openalex,1,"Undoubtedly, the evolution of Generative AI (GenAI) models has been the highlight of digital transformation in the year 2022. As the different GenAI models like ChatGPT and Google Bard continue to foster their complexity and capability, it’s critical to understand its consequences from a cybersecurity perspective. Several instances recently have de...",https://doi.org/10.1109/access.2023.3300381
71,rec_000011,Automating Code Review Activities by Large-Scale Pre-training,2022,Li,arXiv (Cornell University),openalex,1,"Code review is an essential part to software development lifecycle since it aims at guaranteeing the quality of codes. Modern code review activities necessitate developers viewing, understanding and even running the programs to assess logic, functionality, latency, style and other factors. It turns out that developers have to spend far too much tim...",https://doi.org/10.48550/arxiv.2203.09095
72,rec_000009,Towards Automating Code Review Activities,2021,Tufano,NaN,openalex,1,"Code reviews are popular in both industrial and open source projects. The benefits of code review

## 4. Downloaded PDF resources map

This step creates a `resources` directory inside the project, with a `pdfs` subdirectory and CSV and Markdown manifests. It includes only open-access PDFs successfully downloaded by SLR-Engine, retaining the original source path and resolver provenance.

In [7]:
import os
import shutil

resources_root = project_root / 'resources'
pdfs_root = resources_root / 'pdfs'
resources_root.mkdir(exist_ok=True)
pdfs_root.mkdir(exist_ok=True)

with sqlite3.connect(project_root / 'project.db') as connection:
    pdf_resources = pd.read_sql_query(
        """
        SELECT
            records.canonical_id,
            records.title,
            records.year,
            records.first_author,
            records.venue,
            records.doi,
            records.oa_status,
            records.license,
            downloads.resolver_source,
            downloads.file_path AS original_file_path
        FROM downloads
        JOIN records ON records.id = downloads.record_id
        WHERE downloads.status = 'success'
          AND LOWER(downloads.file_format) = 'pdf'
        ORDER BY records.year DESC, records.title
        """,
        connection,
    )

mapped_rows = []
for paper in pdf_resources.to_dict('records'):
    source_path = project_root / paper['original_file_path']
    destination_path = pdfs_root / f"{paper['canonical_id']}.pdf"

    if not source_path.exists() or source_path.read_bytes()[:5] != b'%PDF-':
        print(f"Skipped invalid or missing PDF: {source_path}")
        continue

    if not destination_path.exists():
        try:
            os.link(source_path, destination_path)
        except OSError:
            shutil.copy2(source_path, destination_path)

    paper['resource_pdf'] = str(destination_path.relative_to(project_root))
    mapped_rows.append(paper)

pdf_manifest = pd.DataFrame(mapped_rows)
csv_path = resources_root / 'pdf_resources.csv'
markdown_path = resources_root / 'pdf_resources.md'
pdf_manifest.to_csv(csv_path, index=False)

markdown_columns = [
    'canonical_id', 'title', 'year', 'first_author', 'doi',
    'resolver_source', 'resource_pdf', 'original_file_path',
]
markdown_table = pdf_manifest.reindex(columns=markdown_columns).fillna('')
markdown_lines = [
    '# Downloaded PDF resources',
    '',
    'This manifest contains only successful open-access PDF downloads.',
    '',
]
if markdown_table.empty:
    markdown_lines.append('No PDFs have been downloaded yet.')
else:
    markdown_lines.extend([
        '| ' + ' | '.join(markdown_columns) + ' |',
        '| ' + ' | '.join(['---'] * len(markdown_columns)) + ' |',
    ])
    for row in markdown_table.itertuples(index=False, name=None):
        values = [str(value).replace('|', '\\|').replace('\n', ' ') for value in row]
        markdown_lines.append('| ' + ' | '.join(values) + ' |')
markdown_path.write_text('\n'.join(markdown_lines) + '\n', encoding='utf-8')

print(f'Resources folder: {resources_root}')
print(f'PDF folder: {pdfs_root}')
print(f'CSV manifest: {csv_path}')
print(f'Markdown manifest: {markdown_path}')
print(f'Mapped PDFs: {len(pdf_manifest)}')

if pdf_manifest.empty:
    print('No open-access PDFs have been downloaded yet.')
    print('After committing title/abstract include decisions, run:')
    print(f'  {sys.executable} scripts/05_resolve_oa.py --project {project_id}')
    print(f'  {sys.executable} scripts/06_download.py --project {project_id}')
else:
    display(pdf_manifest)

Resources folder: D:\OneDrive - Hogeschool Rotterdam\1_CURRENT_CODE\FLOWISE_DEEP_RESEARCH_GOOGLE\GOOGLE_API_KEY\GOOGLE_SCHOLAR\SLR-Engine\projects\Willma_SLR\resources
PDF folder: D:\OneDrive - Hogeschool Rotterdam\1_CURRENT_CODE\FLOWISE_DEEP_RESEARCH_GOOGLE\GOOGLE_API_KEY\GOOGLE_SCHOLAR\SLR-Engine\projects\Willma_SLR\resources\pdfs
CSV manifest: D:\OneDrive - Hogeschool Rotterdam\1_CURRENT_CODE\FLOWISE_DEEP_RESEARCH_GOOGLE\GOOGLE_API_KEY\GOOGLE_SCHOLAR\SLR-Engine\projects\Willma_SLR\resources\pdf_resources.csv
Markdown manifest: D:\OneDrive - Hogeschool Rotterdam\1_CURRENT_CODE\FLOWISE_DEEP_RESEARCH_GOOGLE\GOOGLE_API_KEY\GOOGLE_SCHOLAR\SLR-Engine\projects\Willma_SLR\resources\pdf_resources.md
Mapped PDFs: 0
No open-access PDFs have been downloaded yet.
After committing title/abstract include decisions, run:
  C:\Users\PROMET02\anaconda3\envs\prisma-env\python.exe scripts/05_resolve_oa.py --project Willma_SLR
  C:\Users\PROMET02\anaconda3\envs\prisma-env\python.exe scripts/06_download.

## 4. SURF AI-HUB advisory recommendation for one retrieved record

This section deduplicates the retrieved evidence, prepares a screening batch, discovers the models visible to the configured SURF AI-HUB API key, and selects the strongest available Qwen 2.5 Instruct model with at least 32B parameters. It sends one candidate record for an advisory recommendation only. The response is displayed for human assessment and is never committed as a screening decision.

In [8]:
# Prepare an auditable screening batch, then obtain one advisory result from SURF AI-HUB.
from dotenv import load_dotenv
import os
import re
import requests

with sqlite3.connect(project_root / 'project.db') as connection:
    existing_decision_count = connection.execute(
        'SELECT COUNT(*) FROM screening WHERE decision IS NOT NULL'
    ).fetchone()[0]

batch_files = sorted((project_root / 'screening').glob('batch_*.jsonl'))
if existing_decision_count:
    print(f'Reusing {existing_decision_count} existing screening decision(s); deduplication is skipped.')
elif batch_files:
    print('Reusing an existing unreviewed screening batch; deduplication is skipped.')
else:
    dedup_result = subprocess.run(
        [
            sys.executable,
            'scripts/03_dedup.py',
            '--project',
            project_id,
            '--acknowledge-warnings',
        ],
        cwd=repo_root,
        text=True,
        capture_output=True,
    )
    print(dedup_result.stdout)
    if dedup_result.returncode:
        print(dedup_result.stderr)
        raise RuntimeError(f'Deduplication failed with exit code {dedup_result.returncode}.')

    batch_result = subprocess.run(
        [
            sys.executable,
            'scripts/04_screen_prep.py',
            '--project',
            project_id,
            '--batch-size',
            '5',
        ],
        cwd=repo_root,
        text=True,
        capture_output=True,
    )
    print(batch_result.stdout)
    if batch_result.returncode:
        print(batch_result.stderr)
        raise RuntimeError(f'Screening-batch preparation failed with exit code {batch_result.returncode}.')
    batch_files = sorted((project_root / 'screening').glob('batch_*.jsonl'))
if not batch_files:
    raise FileNotFoundError('No screening batch was created from the retrieved evidence.')

with batch_files[0].open(encoding='utf-8') as batch_file:
    candidate = json.loads(next(batch_file))

candidate_summary = pd.DataFrame(
    [
        {
            'title': candidate.get('title'),
            'year': candidate.get('year'),
            'first_author': candidate.get('first_author'),
            'source_url': candidate.get('url'),
        }
    ]
)
display(candidate_summary)

# Configuration follows the working WILLMA stress-test contract.
SURF_DISCOVERY_URL = 'https://api.willma.surf.nl/v0/sequences'
SURF_CHAT_URL = 'https://willma.surf.nl/api/v0/chat/completions'
CONNECT_TIMEOUT_SECONDS = 30
READ_TIMEOUT_SECONDS = 180
MINIMUM_QWEN_PARAMETERS_B = 32

env_path = repo_root / '.env'
if not env_path.exists():
    raise FileNotFoundError(f'Missing SURF AI-HUB configuration file: {env_path}')

load_dotenv(env_path, override=True)
surf_api_key = os.getenv('SURF_AI_HUB_API_KEY', '').strip()
if not surf_api_key:
    raise RuntimeError('SURF_AI_HUB_API_KEY is missing from .env.')

surf_headers = {
    'X-API-KEY': surf_api_key,
    'Content-Type': 'application/json',
}

discovery_response = requests.get(
    SURF_DISCOVERY_URL,
    headers=surf_headers,
    timeout=(CONNECT_TIMEOUT_SECONDS, READ_TIMEOUT_SECONDS),
)
print(f'SURF AI-HUB model discovery status: {discovery_response.status_code}')
if discovery_response.status_code in {401, 403}:
    raise RuntimeError(
        'SURF AI-HUB rejected SURF_AI_HUB_API_KEY. Verify that the key is current and '
        'authorized for the WILLMA model-discovery endpoint.'
    )
discovery_response.raise_for_status()
models_payload = discovery_response.json()
if not isinstance(models_payload, list):
    raise RuntimeError('SURF AI-HUB model discovery returned an unexpected payload.')

models_df = pd.DataFrame(models_payload)
if 'sequence_type' not in models_df.columns:
    models_df['sequence_type'] = 'unknown'
if 'name' not in models_df.columns:
    raise RuntimeError('SURF AI-HUB model discovery did not return model names.')

def qwen_parameter_count_b(model_name):
    match = re.search(r'(?<!\d)(\d+)\s*b(?:\b|[-_])', str(model_name), flags=re.IGNORECASE)
    return int(match.group(1)) if match else None

def is_eligible_qwen_model(model_name, sequence_type):
    normalized_name = str(model_name).casefold()
    return (
        str(sequence_type).casefold() == 'text'
        and 'qwen' in normalized_name
        and '2.5' in normalized_name
        and 'instruct' in normalized_name
        and (qwen_parameter_count_b(model_name) or 0) >= MINIMUM_QWEN_PARAMETERS_B
    )

eligible_models = sorted(
    [
        (qwen_parameter_count_b(row['name']), row['name'])
        for _, row in models_df.iterrows()
        if is_eligible_qwen_model(row['name'], row.get('sequence_type', 'unknown'))
    ],
    reverse=True,
)
available_model_names = set(models_df['name'].dropna().astype(str))
requested_model = os.getenv('SURF_AI_HUB_MODEL', '').strip()

if requested_model:
    if requested_model not in available_model_names:
        raise RuntimeError(f'SURF_AI_HUB_MODEL is not visible to this API key: {requested_model}')
    if not any(model_name == requested_model for _, model_name in eligible_models):
        raise RuntimeError(
            f'SURF_AI_HUB_MODEL must be a Qwen 2.5 Instruct text model with at least '
            f'{MINIMUM_QWEN_PARAMETERS_B}B parameters: {requested_model}'
        )
    selected_model = requested_model
elif eligible_models:
    selected_model = eligible_models[0][1]
else:
    discovered_text_models = models_df.loc[
        models_df['sequence_type'].astype(str).str.casefold().eq('text'),
        'name',
    ].dropna().astype(str).tolist()
    raise RuntimeError(
        'No Qwen 2.5 Instruct model with at least '
        f'{MINIMUM_QWEN_PARAMETERS_B}B parameters is available. '
        f'Discovered text models: {discovered_text_models}'
    )

display(models_df[[column for column in ['id', 'name', 'sequence_type', 'latency_mode'] if column in models_df.columns]])
print(f'Selected SURF AI-HUB model: {selected_model}')

prompt = f'''You are an evidence-screening advisor for a systematic literature review.
Research topic: The Problem With Using AI to Review AI-Written Code.

Apply these criteria:
- Include empirical studies, benchmarks, or systematic evaluations of AI-assisted or AI-driven code review, code analysis, or defect detection.
- Exclude opinion-only work, general code-generation work without review or evaluation, and records with insufficient metadata.

Return exactly these fields:
Decision: include, exclude, or uncertain
Reason: one or two sentences
Criteria: relevant inclusion or exclusion IDs
Human review required: yes

Candidate title: {candidate.get('title', 'Unavailable')}
Candidate abstract: {candidate.get('abstract') or 'Unavailable'}
'''

chat_payload = {
    'model': selected_model,
    'max_tokens': 350,
    'system': (
        'Provide a cautious, evidence-bound recommendation. '
        'Do not claim that the recommendation is a final screening decision.'
    ),
    'messages': [{'role': 'user', 'content': prompt}],
}
chat_response = requests.post(
    SURF_CHAT_URL,
    headers=surf_headers,
    json=chat_payload,
    timeout=(CONNECT_TIMEOUT_SECONDS, READ_TIMEOUT_SECONDS),
)
print(f'SURF AI-HUB advisory status: {chat_response.status_code}')
chat_response.raise_for_status()
chat_result = chat_response.json()

def extract_response_text(payload):
    choices = payload.get('choices', []) if isinstance(payload, dict) else []
    if not choices:
        return str(payload)
    content = choices[0].get('message', {}).get('content', '')
    if isinstance(content, list):
        return ''.join(
            item.get('text', '') if isinstance(item, dict) else str(item)
            for item in content
        )
    return str(content)

usage = chat_result.get('usage', {}) if isinstance(chat_result, dict) else {}
recommendation_table = pd.DataFrame(
    [
        {
            'candidate_title': candidate.get('title'),
            'surf_ai_hub_model': selected_model,
            'surf_advisory': extract_response_text(chat_result),
            'human_decision_required': 'Yes - no screening decision was written',
        }
    ]
)
display(recommendation_table)
display(
    pd.DataFrame(
        [
            {
                'input_tokens': usage.get('prompt_tokens'),
                'output_tokens': usage.get('completion_tokens'),
                'total_tokens': usage.get('total_tokens'),
            }
        ]
    )
)

Records before: 75
Records after:  75
Fuzzy merges:   0 (examined 2 pairs)
Source hits:    75

Next: python scripts/04_screen_prep.py --project Willma_SLR



Batch written: D:\OneDrive - Hogeschool Rotterdam\1_CURRENT_CODE\FLOWISE_DEEP_RESEARCH_GOOGLE\GOOGLE_API_KEY\GOOGLE_SCHOLAR\SLR-Engine\projects\Willma_SLR\screening\batch_001.jsonl
Records:       5
Criteria:      D:\OneDrive - Hogeschool Rotterdam\1_CURRENT_CODE\FLOWISE_DEEP_RESEARCH_GOOGLE\GOOGLE_API_KEY\GOOGLE_SCHOLAR\SLR-Engine\projects\Willma_SLR\screening\_criteria.md

Agent: read skills/slr-engine/SKILL_screening.md and the criteria file, then label each line.
Commit with: python scripts/04b_screen_commit.py --batch D:\OneDrive - Hogeschool Rotterdam\1_CURRENT_CODE\FLOWISE_DEEP_RESEARCH_GOOGLE\GOOGLE_API_KEY\GOOGLE_SCHOLAR\SLR-Engine\projects\Willma_SLR\screening\batch_001.jsonl



,title,year,first_author,source_url
0,Towards automating code review at scale,2021,Hellendoorn,None


SURF AI-HUB model discovery status: 200


,id,name,sequence_type,latency_mode
0,21,openai/whisper-large-v3,stt,on-demand
1,7,openai/whisper-large-v2,stt,always-on
2,29,default-text-large,text,on-demand
3,44,mistralai/Mistral-Small-3.2-24B-Instruct-2506,text,always-on
4,45,openai/gpt-oss-120b,text,always-on
5,47,Qwen/Qwen3-Embedding-8B,embedder,always-on
6,26,Qwen/Qwen2.5-VL-32B-Instruct-AWQ,text,always-on
7,27,Qwen/Qwen2.5-Coder-32B-Instruct-AWQ,text,on-demand
8,49,pyannote/speaker-diarization-3.1,custom-diarization,on-demand
9,50,Sehyo/Qwen3.5-122B-A10B-NVFP4,text,on-demand


Selected SURF AI-HUB model: Qwen/Qwen2.5-VL-32B-Instruct-AWQ


SURF AI-HUB advisory status: 200


,candidate_title,surf_ai_hub_model,surf_advisory,human_decision_required
0,Towards automating code review at scale,Qwen/Qwen2.5-VL-32B-Instruct-AWQ,"**Decision**: include \n**Reason**: The abstract describes an empirical study that evaluates the use of neural models for automating code review at scale. It includes systematic examination of model performance, dataset statistics, and challenges, meeting the inclusion criteria for empirical studies and evaluations. \n**Criteria**: relevant inclusion IDs (empirical studies, benchmarks, systematic evaluations) \n**Human review required**: yes",Yes - no screening decision was written


,input_tokens,output_tokens,total_tokens
0,368,79,447


## 5. Commit reviewed decisions and download open-access PDFs

The first screening batch is labeled against the stated criteria. Only the included, empirically evaluated automated-code-review studies are sent to the open-access resolver and downloader.

In [9]:
reviewed_decisions = {
    1: {
        'decision': 'include',
        'reason': 'Empirical evaluation of neural automated code review at scale; reports dataset, methods, performance challenges, and limitations.',
        'criteria_hit': ['I1', 'I2', 'I3'],
    },
    2: {
        'decision': 'include',
        'reason': 'Empirical study of a fine-tuned LLM for automated code review, evaluating accuracy and comprehensibility.',
        'criteria_hit': ['I1', 'I2', 'I3'],
    },
    3: {
        'decision': 'include',
        'reason': 'Reports an automated code-review tool evaluated in an industrial development workflow.',
        'criteria_hit': ['I1', 'I2', 'I3'],
    },
    4: {
        'decision': 'exclude',
        'reason': 'Broad cybersecurity and privacy discussion; it does not evaluate AI-assisted code review, code analysis, or defect detection.',
        'criteria_hit': ['E1', 'E2'],
    },
    5: {
        'decision': 'include',
        'reason': 'Quantitative and qualitative evaluation of LLM-based automated code review with reported outcomes.',
        'criteria_hit': ['I1', 'I2', 'I3'],
    },
}

batch_path = project_root / 'screening' / 'batch_001.jsonl'
labeled_rows = []
with batch_path.open(encoding='utf-8') as batch_file:
    for line in batch_file:
        record = json.loads(line)
        assessment = reviewed_decisions.get(record['record_id'])
        if assessment:
            record.update(assessment)
        labeled_rows.append(record)

with batch_path.open('w', encoding='utf-8') as batch_file:
    for record in labeled_rows:
        batch_file.write(json.dumps(record, ensure_ascii=False) + '\n')

for command in (
    [
        sys.executable, 'scripts/04b_screen_commit.py',
        '--batch', str(batch_path), '--decided-by', 'agent',
    ],
    [sys.executable, 'scripts/05_resolve_oa.py', '--project', project_id],
    [sys.executable, 'scripts/06_download.py', '--project', project_id],
):
    result = subprocess.run(command, cwd=repo_root, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode:
        print(result.stderr)
        raise RuntimeError(f'Command failed: {" ".join(command)}')

Committed: 5 records from batch_001.jsonl
  include: 4
  exclude: 1
  unsure: 0

Next: another batch with python scripts/04_screen_prep.py --project ...
      or python scripts/05_resolve_oa.py --project ... once screening is done



Resolving OA for 4 records...
Resolved: 3
Queued alternates: 2
Skipped (no OA found): 1

Next: python scripts/06_download.py --project Willma_SLR



  FAIL rec_000001 via openalex: HTTP Error 403: Forbidden
  FAIL rec_000001 via crossref: HTTP Error 403: Forbidden
  FAIL rec_000002 via crossref: HTTP Error 403: Forbidden
  FAIL rec_000005 via openalex: HTTP Error 403: Forbidden
  FAIL rec_000005 via crossref: HTTP Error 403: Forbidden

Success: 0
Failed (all candidates):  3
Not downloaded: 4 — see not_downloaded.csv / not_downloaded.txt

Next: python scripts/07_fulltext_prep.py --project Willma_SLR (or scripts/09_export.py if skipping full-text pass)



In [10]:
arxiv_decisions = {
    52: 'Empirical user study of explainability, trust, and agreement in LLM-assisted code review.',
    53: 'Large-scale empirical analysis of developer responses to agent-generated code review comments.',
    54: 'Empirical study of 1.02 million pull requests measuring AI reviewer effects on efficiency and quality.',
    58: 'Benchmark-based study of AI self-review failure modes when reviewing AI-generated code.',
    59: 'Eye-tracking experiment examining human review behavior for LLM-generated code.',
}

with sqlite3.connect(project_root / 'project.db') as connection:
    candidate_rows = pd.read_sql_query(
        """
        SELECT id AS record_id, canonical_id, title, abstract, year, first_author, venue, doi
        FROM records
        WHERE id IN (52, 53, 54, 58, 59)
        ORDER BY id
        """,
        connection,
    ).to_dict('records')

arxiv_batch_path = project_root / 'screening' / 'batch_002.jsonl'
with arxiv_batch_path.open('w', encoding='utf-8') as batch_file:
    for record in candidate_rows:
        record.update({
            'batch_id': 'batch_002',
            'decision': 'include',
            'reason': arxiv_decisions[record['record_id']],
            'criteria_hit': ['I1', 'I2', 'I3'],
        })
        batch_file.write(json.dumps(record, ensure_ascii=False) + '\n')

for command in (
    [
        sys.executable, 'scripts/04b_screen_commit.py',
        '--batch', str(arxiv_batch_path), '--decided-by', 'agent',
    ],
    [sys.executable, 'scripts/05_resolve_oa.py', '--project', project_id, '--retry-failed'],
    [sys.executable, 'scripts/06_download.py', '--project', project_id, '--retry-failed'],
):
    result = subprocess.run(command, cwd=repo_root, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode:
        print(result.stderr)
        raise RuntimeError(f'Command failed: {" ".join(command)}')

Committed: 5 records from batch_002.jsonl
  include: 5
  exclude: 0
  unsure: 0

Next: another batch with python scripts/04_screen_prep.py --project ...
      or python scripts/05_resolve_oa.py --project ... once screening is done



Cleared 6 download row(s) for retry.
Resolving OA for 9 records...
Resolved: 8
Queued alternates: 2
Skipped (no OA found): 1

Next: python scripts/06_download.py --project Willma_SLR



Re-queued 0 failed record(s).
  FAIL rec_000001 via openalex: HTTP Error 403: Forbidden
  FAIL rec_000001 via crossref: HTTP Error 403: Forbidden
  FAIL rec_000002 via crossref: HTTP Error 403: Forbidden
  FAIL rec_000005 via openalex: HTTP Error 403: Forbidden
  FAIL rec_000005 via crossref: HTTP Error 403: Forbidden
  OK rec_000052.pdf via arxiv (1476 KB)
  OK rec_000053.pdf via arxiv (826 KB)
  OK rec_000054.pdf via arxiv (302 KB)
  OK rec_000058.pdf via arxiv (2048 KB)
  OK rec_000059.pdf via arxiv (1206 KB)

Success: 5
Failed (all candidates):  3
Not downloaded: 4 — see not_downloaded.csv / not_downloaded.txt

Next: python scripts/07_fulltext_prep.py --project Willma_SLR (or scripts/09_export.py if skipping full-text pass)



## 6. Export a data-science evidence overview

This step writes `ai_written_code_review.md` in the project root. It records the literal searches, source coverage, screening and download audit, a transparent relevance ranking, and APA-style references.

In [25]:
from datetime import datetime, timezone
import json
import re

# Render a compact, auditable Markdown overview from the project database.
report_path = project_root / 'ai_written_code_review.md'
report_terms = (
    'code review', 'automated code review', 'ai code review', 'llm',
    'large language model', 'generated code', 'code generation',
    'agent-generated', 'self-review', 'human oversight',
)

def markdown_text(value):
    return str(value or '').replace('|', '\\|').replace('\n', ' ').strip()

def metadata_text(value):
    text = '' if pd.isna(value) else str(value).strip()
    return '' if text.lower() == 'nan' else text

def apa_initials(given):
    return ' '.join(f'{part[0]}.' for part in re.findall(r"[A-Za-z]+", metadata_text(given)))

def apa_authors(authors_json, fallback):
    try:
        authors = json.loads(metadata_text(authors_json))
    except json.JSONDecodeError:
        authors = []
    names = [
        f"{metadata_text(author.get('family'))}, {apa_initials(author.get('given'))}".rstrip(', ')
        for author in authors[:20]
        if metadata_text(author.get('family'))
    ]
    if len(authors) > 20:
        names = names[:19] + ['...'] + names[-1:]
    if not names:
        return metadata_text(fallback) or 'Unknown author'
    if len(names) == 1:
        return names[0]
    return ', '.join(names[:-1]) + ', & ' + names[-1]

def markdown_table(columns, rows):
    lines = [
        '| ' + ' | '.join(columns) + ' |',
        '| ' + ' | '.join(['---'] * len(columns)) + ' |',
    ]
    for row in rows:
        lines.append('| ' + ' | '.join(markdown_text(value) for value in row) + ' |')
    return '\n'.join(lines)

with sqlite3.connect(project_root / 'project.db') as connection:
    records = pd.read_sql_query(
        """
        SELECT
            r.id, r.canonical_id, r.title, r.abstract, r.year, r.authors_json, r.first_author,
            r.venue, r.doi, r.url, bm.container_title, bm.volume, bm.issue, bm.pages, bm.publisher,
            GROUP_CONCAT(DISTINCT sh.source) AS sources,
            s.decision, s.reason, s.criteria_hit,
            MAX(CASE WHEN d.status = 'success' AND LOWER(d.file_format) = 'pdf' THEN 1 ELSE 0 END) AS has_pdf
        FROM records r
        LEFT JOIN source_hits sh ON sh.record_id = r.id
        LEFT JOIN screening s ON s.record_id = r.id AND s.pass = 'title_abstract'
        LEFT JOIN bibliographic_metadata bm ON bm.record_id = r.id
        LEFT JOIN downloads d ON d.record_id = r.id
        GROUP BY r.id
        """,
        connection,
    )
    source_counts = pd.read_sql_query(
        "SELECT source, COUNT(DISTINCT record_id) AS records FROM source_hits GROUP BY source ORDER BY records DESC, source",
        connection,
    )
    screening_counts = pd.read_sql_query(
        "SELECT decision, COUNT(*) AS records FROM screening WHERE pass = 'title_abstract' GROUP BY decision ORDER BY decision",
        connection,
    )
    download_counts = pd.read_sql_query(
        "SELECT status, COUNT(DISTINCT record_id) AS records FROM downloads GROUP BY status ORDER BY status",
        connection,
    )

records['title'] = records['title'].fillna('Untitled record')
records['abstract'] = records['abstract'].fillna('')
records['matched_terms'] = records.apply(
    lambda record: [
        term for term in report_terms
        if term in f"{record['title']} {record['abstract']}".lower()
    ],
    axis=1,
)
records['relevance_score'] = records['matched_terms'].str.len()
records.loc[records['decision'].eq('include'), 'relevance_score'] += 4
records.loc[records['has_pdf'].eq(1), 'relevance_score'] += 2
records['ranking_reason'] = records.apply(
    lambda record: '; '.join(
        ([f"matched: {', '.join(record['matched_terms'])}"] if record['matched_terms'] else [])
        + (['title/abstract included'] if record['decision'] == 'include' else [])
        + (['open-access PDF downloaded'] if record['has_pdf'] else [])
    ) or 'metadata-only match',
    axis=1,
)
ranked_records = records.sort_values(
    ['relevance_score', 'has_pdf', 'year', 'title'],
    ascending=[False, False, False, True],
).reset_index(drop=True)
ranked_records.index += 1

query_files = {
    'OpenAlex': 'openalex.txt',
    'arXiv': 'arxiv.txt',
    'Semantic Scholar': 'semantic_scholar.txt',
    'Crossref': 'crossref.json',
}
queries = {
    source: (project_root / 'queries' / filename).read_text(encoding='utf-8').strip()
    for source, filename in query_files.items()
}

source_rows = source_counts[['source', 'records']].itertuples(index=False, name=None)
screening_rows = screening_counts[['decision', 'records']].itertuples(index=False, name=None)
download_rows = download_counts[['status', 'records']].itertuples(index=False, name=None)
ranking_rows = []
for rank, record in ranked_records.head(20).iterrows():
    author = apa_authors(record['authors_json'], record['first_author'])
    year = int(record['year']) if pd.notna(record['year']) else 'n.d.'
    sources = metadata_text(record['sources']) or 'unknown source'
    ranking_rows.append((
        rank,
        f"**{record['title']}**<br>{author} ({year})",
        f"**Score: {record['relevance_score']}**<br>{record['ranking_reason']}<br>Sources: {sources}",
    ))

reference_records = ranked_records.copy()
reference_records['reference_author'] = reference_records['first_author'].map(metadata_text).str.casefold()
reference_records = reference_records.sort_values(['reference_author', 'year', 'title'], na_position='last')
reference_lines = []
for _, record in reference_records.iterrows():
    author = apa_authors(record['authors_json'], record['first_author'])
    year = int(record['year']) if pd.notna(record['year']) else 'n.d.'
    venue_name = metadata_text(record['venue'])
    container_title = metadata_text(record['container_title']) or venue_name
    publisher = metadata_text(record['publisher'])
    volume = metadata_text(record['volume'])
    issue = metadata_text(record['issue'])
    pages = metadata_text(record['pages'])
    doi = metadata_text(record['doi'])
    url = metadata_text(record['url'])
    venue = f" <em>{container_title}</em>" if container_title else ''
    if volume:
        venue += f", <em>{volume}</em>"
    if issue:
        venue += f"({issue})"
    if pages:
        venue += f", {pages}"
    if not venue and publisher:
        venue = f" {publisher}"
    if venue:
        venue += '.'
    doi_url = f"https://doi.org/{doi}"
    doi_link = f" <a href=\"{doi_url}\">{doi_url}</a>" if doi else ''
    url_link = f" <a href=\"{url}\">{url}</a>" if not doi and url else ''
    reference_lines.append(
        f"<li>{author} ({year}). {record['title']}.{venue}{doi_link or url_link}</li>"
    )

report_lines = [
    '# AI-Written Code Review: Evidence Overview',
    '',
    '## Contents',
    '',
    '- [Scope](#scope)',
    '- [Provenance and Notebook Development](#provenance-and-notebook-development)',
    '- [Search Method](#search-method)',
    '- [Relevance Ranking](#relevance-ranking)',
    '- [Relevant Sources Found Through SLR](#relevant-sources-found-through-slr)',
    '- [Reproducibility Notes](#reproducibility-notes)',
    '- [Appendix A. Verantwoording en correcties](#appendix-a-verantwoording-en-correcties)',
    '',
    f"Generated: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}",
    '',
    '## Scope',
    '',
    '**Topic:** An automated, AI-assisted systematic literature review addressing:',
    '',
    f"**{topic}**",
    '',
    '**Aim:** To synthesize empirical evidence on the reliability, risks, and forms of human oversight required when AI systems evaluate AI-generated code.',
    '',
    'This report presents a reproducible, auditable overview of the current [SLR-Engine](https://github.com/tuirk/SLR-Engine) project. It distinguishes retrieved evidence, AI-assisted screening recommendations, and the transparent ranking heuristic used to support inspection; it does not treat automated output as a substitute for scholarly judgement.',
    '',
    '## Provenance and Notebook Development',
    '',
    'This notebook is based on [SLR-Engine](https://github.com/tuirk/SLR-Engine), an auditable workflow for systematic literature reviews. It uses a SURF AI-HUB Qwen 2.5 Instruct model only for bounded screening advice; final decisions remain subject to human review.',
    '',
    'The notebook persists literal source queries, retrieves and deduplicates records, prepares screening batches, records reviewed decisions, resolves lawful open-access copies, validates downloaded PDFs, and exports this evidence overview. Source provenance, decisions, download outcomes, and ranking signals remain database-backed so the workflow can be rerun and extended without treating model output as final scholarly judgement.',
    '',
    '## Search Method',
    '',
    'The notebook searched OpenAlex, Crossref, arXiv, and Semantic Scholar with a cap of 25 source hits per source. Any source failure remains an audit limitation rather than evidence of no results.',
    '',
    '### Literal Queries',
    '',
]
for source, query in queries.items():
    report_lines.extend([f'**{source}**', '', '```text', query, '```', ''])
report_lines.extend([
    '### Retrieval and Processing Audit',
    '',
    f"Canonical records after deduplication: **{len(records)}**.",
    '',
    markdown_table(['Source', 'Distinct records'], source_rows),
    '',
    'Title/abstract screening decisions are recorded with stated reasons and criteria in the project database.',
    '',
    markdown_table(['Decision', 'Records'], screening_rows),
    '',
    'Download outcomes represent lawful open-access retrieval attempts only; publisher-blocked or closed content is not bypassed.',
    '',
    markdown_table(['Download status', 'Records'], download_rows),
    '',
    '## Relevance Ranking',
    '',
    'Ranking is an inspection aid, not an inclusion decision. Score = number of matched topic terms + 4 for a committed title/abstract include decision + 2 for a verified downloaded open-access PDF. The rationale is listed for each paper.',
    '',
    '<div style="font-size: smaller;">',
    markdown_table(['Rank', 'Paper', 'Evidence'], ranking_rows),
    '</div>',
    '',
    'The table is limited to the 20 highest-ranked records; the complete metadata remains in the project database and source inventory below.',
    '',
    '## Relevant Sources Found Through SLR',
    '',
    'This inventory contains the deduplicated records retrieved by the SLR workflow from OpenAlex, Crossref, arXiv, and Semantic Scholar. Entries use APA 7-style author, date, title, source, volume, issue, pages, and DOI/URL formatting when metadata is available. DOI URLs are retained. DOI-bearing records are enriched from Crossref and cached in the local audit database; records without a DOI or unavailable Crossref metadata retain the source fields provided by the original search result. Appearing here does not by itself indicate a final eligibility decision, methodological-quality assessment, or full-text review.',
    '',
    '<ol style="font-size: smaller;">',
    '',
] + reference_lines + [
    '</ol>',
    '',
    '## Reproducibility Notes',
    '',
    '- Query files, source hits, deduplication, screening decisions, download attempts, and local PDF paths are retained in the project directory and SQLite audit database.',
    '- SURF AI-HUB advice is model-selected at runtime and displayed without writing a decision; the selected model and request scope should be recorded before any publication use.',
    '- Results should be reviewed before publication, especially metadata-only records and screening decisions.',
    '',
    '## Appendix A. Verantwoording en correcties',
    '',
    'Deze rapportage is opgesteld met ondersteuning van generatieve AI. De auteur blijft verantwoordelijk voor de onderzoeksopzet, de controle van de feiten, de selectie van bronnen, de inhoudelijke interpretatie en de uiteindelijke tekst. AI-uitvoer is gebruikt als ondersteuning en niet als zelfstandig bewijs of als vervanging van menselijk oordeel.',
    '',
    '### Gebruikte AI-ondersteuning',
    '',
    markdown_table(['Tool of component', 'Rol in deze rapportage', 'Beheersmaatregel'], [
        ('SURF AI-HUB Qwen 2.5 Instruct model (minimaal 32B)', 'Begrensd advies bij screening van een kandidaatrecord.', 'Model is runtime geselecteerd; advies schrijft geen beslissing en vereist menselijke beoordeling.'),
        ('SLR-Engine workflow', 'Herleidbare zoek-, deduplicatie-, screening- en exportstappen.', 'Queries, bronhits, beslissingen en downloaduitkomsten blijven lokaal auditbaar.'),
    ]),
    '',
    '### Juridische en ethische uitgangspunten',
    '',
    '- **Artikel 5 AVG:** doelbinding en dataminimalisatie zijn relevant voor zover in de workflow persoonsgegevens worden verwerkt.',
    '- **Artikel 12 AVG:** informatie over verwerking van persoonsgegevens moet duidelijk en toegankelijk zijn wanneer betrokkenen moeten worden geinformeerd.',
    '- **Artikel 13 AI Act:** bevat transparantieverplichtingen voor hoog-risico-AI-systemen; de bepaling is alleen van toepassing wanneer een systeem onder die categorie valt.',
    '- **Artikel 50 AI Act:** bevat specifieke transparantieverplichtingen, onder meer rond AI-interactie en bepaalde synthetische inhoud. De toepasselijkheid hangt af van de concrete inzet en rol van het AI-systeem.',
    '',
    'Deze appendix is een transparantieverklaring voor deze rapportage en geen juridisch advies. Bij verwerking van persoonsgegevens of inzet in een gereguleerde context is aanvullende toetsing nodig.',
    '',
    '## Appendix B. Reproducible Procedure and Code-Cell Explanation',
    '',
    'This appendix describes the executable V04 workflow as implemented. A data scientist can reproduce the local evidence workflow without treating the AI response as a final scholarly decision.',
    '',
    '### Prerequisites',
    '',
    '1. Open `SLR_Engine_V04_SURF_AI_HUB_Demo.ipynb` from the SLR-Engine repository root.',
    '2. Select the `prisma-env` Jupyter kernel and install the repository dependencies in that environment.',
    '3. Create a repository-root `.env` file containing `SURF_AI_HUB_API_KEY=<key>`. Optionally set `SURF_AI_HUB_MODEL=<visible eligible model name>` to pin a model; otherwise the notebook selects the largest eligible model.',
    '4. Ensure outbound HTTPS access to OpenAlex, Crossref, arXiv, Semantic Scholar, `https://api.willma.surf.nl`, and `https://willma.surf.nl`. Search results and visible models vary by date, network, API key, and source availability.',
    '5. Run the notebook in order. Do not rerun deduplication after screening decisions exist; the notebook intentionally reuses existing batches in that case.',

    '### Code Cells and Their Exact Role',

    '1. **Kernel verification.** Prints the executable, Python version, and environment prefix, then raises an error unless the active interpreter belongs to `prisma-env`. This prevents a notebook from silently running in the base Conda environment.',
    '2. **Repository verification.** Confirms `README.md`, `scripts/`, `slr_engine/`, and `tests/` exist below the current working directory, then runs `pytest -q` with the active interpreter. It stops on test failure before project data is changed.',
    '3. **Project and protocol initialization.** Creates or reuses `projects/Willma_SLR`, then writes `project.yaml` with the topic, aim, research questions, inclusion criteria I1-I3, exclusion criteria E1-E3, seed placeholders, and `agent` as the advisory provider.',
    '4. **Protocol display.** Renders the saved protocol as a pandas table. It is a review checkpoint: verify the research question and eligibility criteria before any retrieval.',
    '5. **Bounded literature search.** Writes literal query files under `projects/Willma_SLR/queries/`, calls `scripts/02_search_open.py` for OpenAlex, Crossref, arXiv, and Semantic Scholar with a 25-record-per-source cap, then queries `project.db` to display the first ten provenance-backed records.',
    '6. **Retrieved-evidence inspection.** Reads all database records and source hits, calculates a transparent count of topic-term signals, and displays per-source, per-year, and per-paper tables. This score only prioritizes inspection; it is not a screening decision.',
    '7. **PDF resources map.** Reads successful PDF downloads, validates the `%PDF-` signature, hard-links or copies each valid file to `projects/Willma_SLR/resources/pdfs/`, and writes CSV and Markdown manifests with source and resolver provenance.',
    '8. **SURF AI-HUB advisory call.** Reuses an existing unreviewed screening batch or runs `03_dedup.py` and `04_screen_prep.py`. It sends one candidate title and abstract to SURF AI-HUB, discovers models through `/v0/sequences`, requires a text Qwen 2.5 Instruct model with at least 32B parameters, and posts a bounded advisory prompt to `/api/v0/chat/completions`. The displayed response, selected model, and token usage are advisory evidence only; this cell does not write a screening decision.',
    '9. **First reviewed batch and downloads.** Writes demo labels for `batch_001.jsonl`, commits them with `04b_screen_commit.py`, resolves lawful open-access locations with `05_resolve_oa.py`, and downloads with `06_download.py`. Replace the example labels and reasons with independently reviewed judgements before using this as a study dataset. The `--decided-by agent` value is provenance metadata, not evidence that an AI may make final decisions.',
    '10. **Additional reviewed batch.** Selects the five hard-coded database record IDs 52, 53, 54, 58, and 59, writes `batch_002.jsonl`, commits the supplied include labels, then retries OA resolution and downloads. These IDs are specific to the saved demo database and must be replaced by stable canonical identifiers or a human-selected query in a fresh reproduction.',
    '11. **Evidence report export.** Reads the SQLite audit tables and query files, calculates the report ranking, creates an HTML-compatible APA-style reference list from available metadata, and writes `projects/Willma_SLR/ai_written_code_review.md`. Its ranking is exactly: matched topic-term count + 4 for a committed include decision + 2 for a verified downloaded PDF.',
    '12. **Compact ranking refresh.** Replaces only the generated report ranking section using the in-memory ranked records. It is a formatting refresh and must be run after the report-export cell in the same kernel.',
    '13. **Standalone APA-style export.** Reads `project.db` independently and writes `projects/Willma_SLR/references_apa.md`. It preserves source-limited first-author metadata and must not be interpreted as a fully verified APA bibliography.',

    '### AI and SLR Decision Boundary',

    '- The AI service receives one selected candidate title and abstract, not repository source code and not an entire screening batch.',
    '- The model prompt requires `Decision`, `Reason`, `Criteria`, and `Human review required: yes`; its response is displayed but never inserted into the `screening` table by the advisory cell.',
    '- SLR inclusion and exclusion decisions must be reviewed against I1-I3 and E1-E3, documented with reasons, and retained in `project.db` and JSONL batch files.',
    '- Downloading is limited to resolver-discovered open-access copies; failed or blocked downloads remain audit outcomes and must not be bypassed.',
    '- Reproducibility means preserving the notebook version, package versions, literal query files, source timestamps, project database, batch JSONL files, model name, prompt, raw response, and human-decision provenance. It does not mean that a future source search or model call will return identical results.',

    '### Expected Outputs',

    '- `projects/Willma_SLR/project.yaml`: protocol and configuration.',
    '- `projects/Willma_SLR/project.db`: records, source hits, screening, and download audit trail.',
    '- `projects/Willma_SLR/queries/`: literal source queries.',
    '- `projects/Willma_SLR/screening/`: prepared and reviewed JSONL batches.',
    '- `projects/Willma_SLR/resources/`: verified PDF map and manifests when lawful PDFs are available.',
    '- `projects/Willma_SLR/ai_written_code_review.md` and `references_apa.md`: human-readable evidence and source-metadata exports.',

])

# Embed the executable notebook source so this report remains self-contained.
notebook_path = repo_root / 'SLR_Engine_V04_SURF_AI_HUB_Demo.ipynb'
notebook_document = json.loads(notebook_path.read_text(encoding='utf-8'))
report_lines.extend([
    '### Verbatim Code by Cell',
    '',
    'The following blocks are copied directly from the notebook source at report-generation time. They contain no secret values; API keys are read from `.env` at runtime.',
    '',
])
code_cell_number = 0
for notebook_cell in notebook_document['cells']:
    if notebook_cell.get('cell_type') != 'code':
        continue
    code_cell_number += 1
    report_lines.extend([
        f'#### Code Cell {code_cell_number}',
        '',
        '```python',
        ''.join(notebook_cell.get('source', [])).rstrip(),
        '```',
        '',
    ])
report_path.write_text('\n'.join(report_lines), encoding='utf-8')
print(f'Wrote report: {report_path}')
print(f'Ranked records: {len(ranked_records)}')
display(ranked_records.head(20)[['title', 'year', 'relevance_score', 'ranking_reason']])

Wrote report: D:\OneDrive - Hogeschool Rotterdam\1_CURRENT_CODE\FLOWISE_DEEP_RESEARCH_GOOGLE\GOOGLE_API_KEY\GOOGLE_SCHOLAR\SLR-Engine\projects\Willma_SLR\ai_written_code_review.md
Ranked records: 75


,title,year,relevance_score,ranking_reason
1,"Same Scrutiny, More Time: Eye Tracking Insights into Reviewing LLM-Labelled Code",2026,10,"matched: code review, llm, large language model, generated code; title/abstract included; open-access PDF downloaded"
2,When AI Reviews Its Own Code: Recursive Self-Training Collapse in Code LLMs,2026,10,"matched: code review, llm, generated code, self-review; title/abstract included; open-access PDF downloaded"
3,"""Go Home Copilot, You're Drunk"": Understanding Developer Responses to Agent-Generated Code Review Comments",2026,9,"matched: code review, generated code, agent-generated; title/abstract included; open-access PDF downloaded"
4,Evaluating the Impact of Explainable AI on Trust in AI-Assisted Code Review,2026,9,"matched: code review, llm, large language model; title/abstract included; open-access PDF downloaded"
5,From Human-Centric to Agentic Code Review: The Impact of Different Generations of Generative AI Technology on Review Quality,2026,9,"matched: code review, llm, large language model; title/abstract included; open-access PDF downloaded"
6,Fine-Tuning Large Language Models to Improve Accuracy and Comprehensibility of Automated Code Review,2024,8,"matched: code review, automated code review, llm, large language model; title/abstract included"
7,Improving Automated Code Reviews: Learning From Experience,2024,7,"matched: code review, automated code review, large language model; title/abstract included"
8,Diggit: Automated code review via software repository mining,2018,6,"matched: code review, automated code review; title/abstract included"
9,"Evaluation of LLM-Based Software Engineering Tools: Practices, Challenges, and Future Directions",2026,5,"matched: code review, automated code review, llm, large language model, code generation"
10,Towards automating code review at scale,2021,5,matched: code review; title/abstract included


In [26]:
from pathlib import Path

import json
import re
import sqlite3

import pandas as pd

repo_root = Path.cwd().resolve()
project_root = repo_root / 'projects' / 'Willma_SLR'
project_db = project_root / 'project.db'
apa_references_path = project_root / 'references_apa.md'

if not project_db.exists():
    raise FileNotFoundError(f'Willma project database is missing: {project_db}')


def reference_value(value):
    text = '' if pd.isna(value) else str(value).strip()
    return '' if text.lower() == 'nan' else text


def apa_initials(given):
    return ' '.join(f'{part[0]}.' for part in re.findall(r"[A-Za-z]+", reference_value(given)))


def apa_authors(authors_json, fallback):
    try:
        authors = json.loads(reference_value(authors_json))
    except json.JSONDecodeError:
        authors = []
    names = [
        f"{reference_value(author.get('family'))}, {apa_initials(author.get('given'))}".rstrip(', ')
        for author in authors[:20]
        if reference_value(author.get('family'))
    ]
    if len(authors) > 20:
        names = names[:19] + ['...'] + names[-1:]
    if not names:
        return reference_value(fallback) or 'Unknown author'
    if len(names) == 1:
        return names[0]
    return ', '.join(names[:-1]) + ', & ' + names[-1]


with sqlite3.connect(project_db) as connection:
    reference_records = pd.read_sql_query(
        '''
        SELECT r.authors_json, r.first_author, r.year, r.title, r.venue, r.doi, r.url,
               bm.container_title, bm.volume, bm.issue, bm.pages, bm.publisher
        FROM records r
        LEFT JOIN bibliographic_metadata bm ON bm.record_id = r.id
        ORDER BY LOWER(COALESCE(r.first_author, '')), r.year, r.title
        ''',
        connection,
    )

apa_references = []
for _, record in reference_records.iterrows():
    author = apa_authors(record['authors_json'], record['first_author'])
    year = int(record['year']) if pd.notna(record['year']) else 'n.d.'
    title = reference_value(record['title']) or 'Untitled record'
    venue = reference_value(record['container_title']) or reference_value(record['venue'])
    volume = reference_value(record['volume'])
    issue = reference_value(record['issue'])
    pages = reference_value(record['pages'])
    publisher = reference_value(record['publisher'])
    doi = reference_value(record['doi'])
    url = reference_value(record['url'])
    source_url = f'https://doi.org/{doi}' if doi else url
    venue_part = f' *{venue}*' if venue else (f' {publisher}' if publisher else '')
    if volume:
        venue_part += f', *{volume}*'
    if issue:
        venue_part += f'({issue})'
    if pages:
        venue_part += f', {pages}'
    if venue_part:
        venue_part += '.'
    source_part = f' {source_url}' if source_url else ''
    apa_references.append(f'{author} ({year}). {title}.{venue_part}{source_part}')

apa_references_path.write_text(
    '\n'.join([
        '# Relevant Sources Found Through SLR (APA 7-Style)',
        '',
        'Generated from retrieved Willma_SLR metadata. Author lists are formatted from stored `authors_json` values and DOI URLs are retained. DOI-bearing records are enriched from Crossref with source, volume, issue, page, and publisher metadata when available; records without a DOI or unavailable Crossref metadata retain their original search fields. Verify entries against the original publication before formal citation or publication use.',
        '',
        *[f'{number}. {reference}' for number, reference in enumerate(apa_references, start=1)],
        '',
    ]),
    encoding='utf-8',
)

print(f'Wrote {len(apa_references)} APA-style references: {apa_references_path}')

Wrote 75 APA-style references: D:\OneDrive - Hogeschool Rotterdam\1_CURRENT_CODE\FLOWISE_DEEP_RESEARCH_GOOGLE\GOOGLE_API_KEY\GOOGLE_SCHOLAR\SLR-Engine\projects\Willma_SLR\references_apa.md


In [ ]:
from pathlib import Path

import json
import re
import sqlite3

import pandas as pd

repo_root = Path.cwd().resolve()
project_root = repo_root / 'projects' / 'Willma_SLR'
project_db = project_root / 'project.db'
apa_references_path = project_root / 'references_apa.md'

if not project_db.exists():
    raise FileNotFoundError(f'Willma project database is missing: {project_db}')


def reference_value(value):
    text = '' if pd.isna(value) else str(value).strip()
    return '' if text.lower() == 'nan' else text


def apa_initials(given):
    return ' '.join(f'{part[0]}.' for part in re.findall(r"[A-Za-z]+", reference_value(given)))


def apa_authors(authors_json, fallback):
    try:
        authors = json.loads(reference_value(authors_json))
    except json.JSONDecodeError:
        authors = []
    names = [
        f"{reference_value(author.get('family'))}, {apa_initials(author.get('given'))}".rstrip(', ')
        for author in authors[:20]
        if reference_value(author.get('family'))
    ]
    if len(authors) > 20:
        names = names[:19] + ['...'] + names[-1:]
    if not names:
        return reference_value(fallback) or 'Unknown author'
    if len(names) == 1:
        return names[0]
    return ', '.join(names[:-1]) + ', & ' + names[-1]


with sqlite3.connect(project_db) as connection:
    reference_records = pd.read_sql_query(
        '''
        SELECT r.authors_json, r.first_author, r.year, r.title, r.venue, r.doi, r.url,
               bm.container_title, bm.volume, bm.issue, bm.pages, bm.publisher
        FROM records r
        LEFT JOIN bibliographic_metadata bm ON bm.record_id = r.id
        ORDER BY LOWER(COALESCE(r.first_author, '')), r.year, r.title
        ''',
        connection,
    )

apa_references = []
for _, record in reference_records.iterrows():
    author = apa_authors(record['authors_json'], record['first_author'])
    year = int(record['year']) if pd.notna(record['year']) else 'n.d.'
    title = reference_value(record['title']) or 'Untitled record'
    venue = reference_value(record['container_title']) or reference_value(record['venue'])
    volume = reference_value(record['volume'])
    issue = reference_value(record['issue'])
    pages = reference_value(record['pages'])
    publisher = reference_value(record['publisher'])
    doi = reference_value(record['doi'])
    url = reference_value(record['url'])
    source_url = f'https://doi.org/{doi}' if doi else url
    venue_part = f' *{venue}*' if venue else (f' {publisher}' if publisher else '')
    if volume:
        venue_part += f', *{volume}*'
    if issue:
        venue_part += f'({issue})'
    if pages:
        venue_part += f', {pages}'
    if venue_part:
        venue_part += '.'
    source_part = f' {source_url}' if source_url else ''
    apa_references.append(f'{author} ({year}). {title}.{venue_part}{source_part}')

apa_references_path.write_text(
    '\n'.join([
        '# Relevant Sources Found Through SLR (APA 7-Style)',
        '',
        'Generated from retrieved Willma_SLR metadata. Author lists are formatted from stored `authors_json` values and DOI URLs are retained. DOI-bearing records are enriched from Crossref with source, volume, issue, page, and publisher metadata when available; records without a DOI or unavailable Crossref metadata retain their original search fields. Verify entries against the original publication before formal citation or publication use.',
        '',
        *[f'{number}. {reference}' for number, reference in enumerate(apa_references, start=1)],
        '',
    ]),
    encoding='utf-8',
)

print(f'Wrote {len(apa_references)} APA-style references: {apa_references_path}')

Wrote 75 APA-style references: D:\OneDrive - Hogeschool Rotterdam\1_CURRENT_CODE\FLOWISE_DEEP_RESEARCH_GOOGLE\GOOGLE_API_KEY\GOOGLE_SCHOLAR\SLR-Engine\projects\Willma_SLR\references_apa.md


## Next actions

1. Add one to three verified seed papers using `scripts/00b_read_seeds.py`.
2. Extract and curate seed vocabulary with `scripts/00c_extract_vocabulary.py`.
3. Replace the query templates with literal, evidence-based query strings and approve them before running `scripts/02_search_open.py`.
4. Screen records through `scripts/04_screen_prep.py` in batches of five or fewer. Record final decisions with provenance and keep the audit trail.